# 03 — Regresioni polinomial, Ridge dhe Lasso

        **Pyetja udhëheqëse:** Kur bëhet një model fleksibël tepër fleksibël?

        Ky notebook është ndërtuar si një mësim i plotë kompjuterik. Rruga kryesore ndjek:
        **problem konkret → model matematikor → zbatim nga parimet e para → bibliotekë profesionale → krahasim → interpretim**.

        ## Objektivat
        - Të dallohet nënpërshtatja nga mbipërshtatja.
- Të ndërtohet matrica polinomiale.
- Të kuptohet regularizimi Ridge/Lasso.
- Të interpretohet kompromisi bias–variance.

        > **Rruga bazë për orën mësimore (45–60 min)** shënohet me seksionet **CORE**.  
        > Seksionet **EXTENDED** dhe **PLAYGROUND** janë për thellim dhe punë të pavarur.

## Pse kjo temë është e nevojshme?

Një model më fleksibël mund të përshtatë më mirë të dhënat e trajnimit dhe, megjithatë, të jetë më i dobët si model shkencor. Kjo është pika ku shfaqet qartë dallimi ndërmjet interpolimit të të dhënave të njohura dhe përgjithësimit te matje të reja.

Do të shohim si kompleksiteti, regularizimi dhe zgjedhja e hiperparametrave ndërveprojnë. Ridge dhe Lasso nuk janë thjesht teknika numerike; ato shprehin një kompromis të kontrolluar ndërmjet përshtatjes dhe stabilitetit.

### Si duhet lexuar ky notebook

Për çdo bllok kodi do të shpjegojmë **pse** kryhet ai hap, çfarë pyetjeje i përgjigjet dhe çfarë duhet të kontrollojmë në dalje. Kodi nuk trajtohet si recetë: çdo grafik dhe çdo metrikë duhet të rikthehet te hipoteza matematikore dhe te interpretimi fizik. Kur një rezultat nuk sillet siç presim, pyetja e parë nuk është “si ta rrisim score-in?”, por “cilën hipotezë të modelit ose të të dhënave po shkelim?”.

### Para se të ekzekutojmë: Mjedisi llogaritës dhe riprodhueshmëria

Ngarkojmë bibliotekat që do të përdoren dhe fiksojmë `RANDOM_STATE = 42`. Ky seed nuk e bën problemin determinist në kuptimin fizik; ai thjesht na lejon të përsërisim të njëjtin eksperiment llogaritës dhe të diskutojmë metodën pa u hutuar nga ndryshime të rastësishme të mostrës.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Mjedisi llogaritës dhe riprodhueshmëria.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import numpy as np
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import pandas as pd
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import matplotlib.pyplot as plt
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from pathlib import Path
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import time

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("03 — Regresioni polinomial, Ridge dhe Lasso")
print("Seed:", RANDOM_STATE)

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 1. Sensor jolinear — CORE

Një sensor real mund të ketë përgjigje jolineare. Modeli polinomial
$$
\hat y=\beta_0+\beta_1x+\cdots+\beta_dx^d
$$
është ende linear në parametra, por kompleksiteti rritet me shkallën $d$.

### Para se të ekzekutojmë: Ndërtimi i një përgjigjeje jolineare

Krijojmë një sensor me përgjigje të butë jolineare dhe shtojmë zhurmë. Duke njohur funksionin e vërtetë mund të dallojmë nënpërshtatjen nga mbipërshtatja dhe të shohim nëse modeli rikuperon strukturën apo ndjek fluktuacionet e rastësishme.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Ndërtimi i një përgjigjeje jolineare.
x = np.linspace(-2.5, 2.5, 70)
y_true = 1.0 + 0.7*x - 0.25*x**2 + 0.08*x**3
# Gjenerojmë të dhëna të kontrolluara që na lejojnë të dimë strukturën që duhet të zbulojë modeli.
y = y_true + rng.normal(0, 0.7, len(x))

# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig, ax = plt.subplots()
ax.scatter(x, y, s=18, label="matje")
ax.plot(x, y_true, label="përgjigjja reale")
ax.legend(); ax.set_xlabel("Sinjali hyrës"); ax.set_ylabel("Përgjigjja")
ax.set_title("Kalibrim jolinear")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 2. Nënpërshtatje → përshtatje e mirë → mbipërshtatje — CORE

### Para se të ekzekutojmë: Tre nivele kompleksiteti

Vendosim në të njëjtin grafik një model tepër të thjeshtë, një model të arsyeshëm dhe një model tepër fleksibël. Ky krahasim vizual është më i fuqishëm se përkufizimet abstrakte të bias dhe variance, sepse shohim drejtpërdrejt se çfarë “mëson” secili model.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Tre nivele kompleksiteti.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.preprocessing import PolynomialFeatures
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.linear_model import LinearRegression
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.pipeline import make_pipeline

degrees = [1, 3, 14]
grid = np.linspace(x.min(), x.max(), 400)

# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig, ax = plt.subplots()
ax.scatter(x, y, s=16, label="të dhëna")
for d in degrees:
    # Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
    m = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(x[:,None], y)
    # Llogarisim parashikimet mbi të dhëna që nuk përdoren për të ndryshuar parametrat e modelit.
    ax.plot(grid, m.predict(grid[:,None]), label=f"shkalla {d}")
ax.set_ylim(y.min()-3, y.max()+3)
ax.legend(); ax.set_title("Tre regjime të kompleksitetit")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

**Interpretim.** Shkalla 1 ka bias të madh; shkalla 14 ndjek edhe luhatjet e rastësishme. Modeli i mirë duhet të kapë strukturën e qëndrueshme, jo çdo pikë.

Në një analizë reale ky interpretim duhet të shoqërohet me kontroll të njësive, madhësive karakteristike dhe hipotezave që lejojnë kalimin nga një tregues numerik te një përfundim fizik.

# 3. Gabimi trajnim–validim — CORE

### Para se të ekzekutojmë: Gabimi i trajnimit nuk mjafton

Ndjekim veçmas RMSE në trajnim dhe validim për shkallë të ndryshme. Gabimi i trajnimit priret të bjerë me kompleksitetin; pika interesante është kur gabimi i validimit ndalon së përmirësuari ose rritet. Aty shfaqet mbipërshtatja.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Gabimi i trajnimit nuk mjafton.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.model_selection import train_test_split
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.metrics import mean_squared_error

# Ndajmë të dhënat në mënyrë të riprodhueshme; test-i nuk përdoret për trajnim.
Xtr, Xva, ytr, yva = train_test_split(x[:,None], y, test_size=0.35, random_state=RANDOM_STATE)
deg_grid = range(1, 16)
tr_err, va_err = [], []
for d in deg_grid:
    # Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
    m = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(Xtr, ytr)
    # Llogarisim parashikimet mbi të dhëna që nuk përdoren për të ndryshuar parametrat e modelit.
    tr_err.append(mean_squared_error(ytr, m.predict(Xtr))**0.5)
    # Llogarisim parashikimet mbi të dhëna që nuk përdoren për të ndryshuar parametrat e modelit.
    va_err.append(mean_squared_error(yva, m.predict(Xva))**0.5)

# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig, ax = plt.subplots()
ax.plot(list(deg_grid), tr_err, marker="o", label="trajnim")
ax.plot(list(deg_grid), va_err, marker="o", label="validim")
ax.set_xlabel("Shkalla polinomiale"); ax.set_ylabel("RMSE")
ax.legend(); ax.set_title("Kompleksiteti dhe përgjithësimi")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 4. Ridge: kontroll i amplitudës së koeficientëve — CORE

Ridge minimizon
$$
\|y-X\beta\|_2^2+\lambda\|\beta\|_2^2.
$$
Termi i dytë penalizon koeficientë shumë të mëdhenj dhe e bën zgjidhjen më të qëndrueshme.

### Para se të ekzekutojmë: Rruga e regularizimit

Ndryshojmë $\alpha$ në shumë rende madhësie për të parë si reagojnë koeficientët. Ridge ul normën e tyre gradualisht; Lasso mund të vendosë disa koeficientë saktësisht në zero. Grafiku i kësaj rruge e bën konkret rolin e hiperparametrit.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Rruga e regularizimit.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.linear_model import Ridge, Lasso
alphas = np.logspace(-5, 3, 40)
degree = 10
P = PolynomialFeatures(degree, include_bias=False)
Xp = P.fit_transform(x[:,None])

ridge_norms, lasso_nonzero = [], []
for a in alphas:
    # Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
    r = Ridge(alpha=a).fit(Xp, y)
    ridge_norms.append(np.linalg.norm(r.coef_))
    # Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
    l = Lasso(alpha=a, max_iter=20000).fit(Xp, y)
    lasso_nonzero.append(np.sum(np.abs(l.coef_) > 1e-8))

# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig, ax = plt.subplots()
ax.semilogx(alphas, ridge_norms, marker=".")
ax.set_xlabel("α"); ax.set_ylabel("||β||₂")
ax.set_title("Ridge zvogëlon amplitudën e koeficientëve")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 5. Lasso dhe rrallësia — CORE

### Para se të ekzekutojmë: Rrallësia e Lasso

Numërojmë koeficientët jo-zero sepse Lasso nuk bën vetëm stabilizim; ai mund të prodhojë modele më të rralla. Në interpretim shkencor kjo mund të jetë e dobishme, por zeroja algoritmike nuk duhet ngatërruar me provë se një efekt fizik nuk ekziston.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Rrallësia e Lasso.
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig, ax = plt.subplots()
ax.semilogx(alphas, lasso_nonzero, marker=".")
ax.set_xlabel("α"); ax.set_ylabel("Numri i koeficientëve jo-zero")
ax.set_title("Lasso mund të zerojë koeficientë")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

---
## Deri këtu: materiali bazë për orën mësimore

Në një seancë rreth 60 minuta, zakonisht ndalemi këtu. Seksionet që vijojnë janë për thellim,
ushtrime të pavarura dhe eksperimentim.

# 6. Regularizimi dhe parashikimi — EXTENDED

### Para se të ekzekutojmë: Krahasimi i formës së modeleve

Mbivendosim parashikimet e modelit pa regularizim, Ridge dhe Lasso. Duam të shohim jo vetëm score-in, por edhe formën e funksionit dhe sa fort reagon ai ndaj pikave individuale.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Krahasimi i formës së modeleve.
models = {
    "pa regularizim": LinearRegression(),
    "Ridge α=1": Ridge(alpha=1.0),
    "Lasso α=0.02": Lasso(alpha=0.02, max_iter=20000)
}
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig, ax = plt.subplots()
ax.scatter(x, y, s=14)
for name, est in models.items():
    # Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
    pipe = make_pipeline(PolynomialFeatures(10), est).fit(x[:,None], y)
    # Llogarisim parashikimet mbi të dhëna që nuk përdoren për të ndryshuar parametrat e modelit.
    ax.plot(grid, pipe.predict(grid[:,None]), label=name)
ax.set_ylim(y.min()-3, y.max()+3); ax.legend()
ax.set_title("Efekti i regularizimit")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 7. Aplikim jo-fizik: konsum energjie — EXTENDED

### Para se të ekzekutojmë: Transferimi në një problem tjetër

Shembulli i konsumit të energjisë tregon se ideja e kompleksitetit dhe regularizimit nuk varet nga konteksti fizik. E njëjta matematikë përdoret kur lidhja ndërmjet temperaturës dhe konsumit është jolineare.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Transferimi në një problem tjetër.
T = np.linspace(-5, 35, 120)
# Gjenerojmë të dhëna të kontrolluara që na lejojnë të dimë strukturën që duhet të zbulojë modeli.
energy = 40 + 0.08*(T-20)**2 + rng.normal(0, 2.5, len(T))
# Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
m = make_pipeline(PolynomialFeatures(2), Ridge(alpha=0.1)).fit(T[:,None], energy)
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig, ax = plt.subplots()
ax.scatter(T, energy, s=12)
# Llogarisim parashikimet mbi të dhëna që nuk përdoren për të ndryshuar parametrat e modelit.
ax.plot(T, m.predict(T[:,None]))
ax.set_xlabel("Temperatura e jashtme [°C]"); ax.set_ylabel("Konsum relativ")
ax.set_title("Shembull jashtë Fizikës: energjia e ndërtesës")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# Ushtrime — PLAYGROUND

        ## A. Ushtrime të shkurtra
        1. Gjeni shkallën me RMSE minimal në validim.
2. Krahasoni koeficientët e Ridge për dy vlera të α.
3. Shpjegoni pse Lasso mund të përdoret për përzgjedhje veçorish.
4. Testoni modelin jashtë intervalit të të dhënave dhe diskutoni ekstrapolimin.

        ## B. Ushtrim i plotë
        Ndërtoni një studim Monte Carlo të gabimit trajnim/provë për shkallë 1–12 dhe tregoni shpërndarjen e shkallës së zgjedhur.

        ## C. Playground
        Ndryshoni zhurmën, shkallën polinomiale dhe α.

In [ ]:
# TODO / PLAYGROUND
# Ndryshoni parametrat, shtoni grafikë ose krahasoni modele të tjera.

## Përmbledhje konceptuale e kapitullit

Notebook-u **Regresioni polinomial, Ridge dhe Lasso** duhet të lihet me një ide të qartë: algoritmi është vetëm një hallkë e arsyetimit. Para tij kemi zgjedhjen e përfaqësimit dhe hipotezat mbi të dhënat; pas tij kemi validimin, diagnostikën dhe interpretimin. Nëse një student mund të ekzekutojë të gjitha qelizat por nuk mund të shpjegojë pse secila prej tyre është e nevojshme, atëherë puna llogaritëse nuk është përfunduar ende.

Për studim të mëtejshëm, rikthehuni te shënimet e leksionit për derivimet e plota dhe përdoreni këtë notebook si laborator ku këto ide testohen numerikisht.